# Naive Monte Carlo for the Moran Process

The **Moran process** models evolution on a graph: a population of $N$ individuals
occupies vertices, and a single mutant with fitness $r$ competes against $N-1$ residents
with fitness 1. Under the **Birth-Death (BD)** update rule, each step selects a reproducer
proportional to fitness, then replaces a uniformly chosen neighbor.

The **fixation probability** $\rho$ is the chance that the mutant takes over the entire population.

**Naive MC** (Diaz et al. 2014): simulate every BD step until absorption.
Samples: $\lceil 0.5\, n^2 \ln 16\, /\, \varepsilon^2 \rceil$.
Step limit per run: $16\, \frac{r}{r-1}\, n^4$.

In [ ]:
import sys, pathlib
_root = pathlib.Path.cwd()
if _root.name == 'notebooks': _root = _root.parent
for _d in ['build-python', 'build']:
    if (_root / _d).is_dir(): sys.path.insert(0, str(_root / _d)); break

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import multiprocessing

import moran
from moran import fixation_probability, exact
from moran.graph import CSRGraph, complete_graph, cycle_graph, star_graph, double_star_graph
from moran.algorithms import naive

pd.set_option('display.width', 160)
plt.rcParams.update({'figure.dpi': 130, 'axes.grid': True, 'grid.alpha': 0.3})

COLORS = {'complete': '#1f77b4', 'cycle': '#2ca02c', 'star': '#d62728', 'dbl_star': '#9467bd'}

print(f'libmoran {moran.__version__}')

## 1. Graph zoo

Four population structures used throughout: **complete** ($K_n$, well-mixed),
**cycle** ($C_n$), **star** ($S_n$), and **double star** ($DS_{a,b}$, two hubs each with leaves).

In [ ]:
N_VIS = 12
zoo = {
    'complete':  complete_graph(N_VIS),
    'cycle':     cycle_graph(N_VIS),
    'star':      star_graph(N_VIS),
    'dbl_star':  double_star_graph(N_VIS // 2, N_VIS // 2),
}

rows = []
for name, g in zoo.items():
    ds = g.degree_stats()
    rows.append({'graph': name, 'n': g.num_vertices(), 'm': g.num_edges(),
                 'min_deg': ds.min_degree, 'max_deg': ds.max_degree,
                 'regular': ds.is_regular, 'connected': g.is_connected()})
pd.DataFrame(rows)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

layouts = {
    'complete':  lambda G: nx.shell_layout(G),
    'cycle':     lambda G: nx.circular_layout(G),
    'star':      lambda G: nx.spring_layout(G, seed=42, k=2.0),
    'dbl_star':  lambda G: nx.spring_layout(G, seed=42, k=2.0),
}

for ax, (name, g) in zip(axes, zoo.items()):
    G_nx = g.to_networkx()
    pos = layouts[name](G_nx)
    nx.draw(G_nx, pos, ax=ax, node_size=150, node_color=COLORS[name],
            edge_color='#666', width=0.9, with_labels=False,
            edgecolors='white', linewidths=0.5)
    ds = g.degree_stats()
    label = f'{name}\nn={g.num_vertices()}, m={g.num_edges()}'
    if ds.is_regular:
        label += f'\n{ds.min_degree}-regular'
    else:
        label += f'\ndeg in [{ds.min_degree}, {ds.max_degree}]'
    ax.set_title(label, fontsize=9)

plt.suptitle('Population structure graphs', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

## 2. Exact formulas

Well-mixed: $\rho = (1 - 1/r) / (1 - 1/r^N)$. For $r=1$: $\rho = 1/N$.

**Isothermal theorem**: regular graphs have the same fixation probability as well-mixed
(not "are" well-mixed -- the dynamics differ, but $\rho$ is identical).

In [ ]:
N = 20
r_grid = np.linspace(0.2, 5.0, 300)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
rho = [exact.well_mixed(N, float(r)) for r in r_grid]

ax1.plot(r_grid, rho, 'b-', lw=2)
ax1.axhline(1/N, color='gray', ls='--', alpha=.5, label=f'1/N = {1/N:.3f}')
ax1.axvline(1.0, color='gray', ls=':', alpha=.5)
ax1.set(xlabel='Fitness r', ylabel='\u03c1', title=f'Well-mixed fixation (N={N})', ylim=(0, 1))
ax1.legend()

ax2.plot(r_grid, rho, 'b-', lw=2)
ax2.axhline(1/N, color='gray', ls='--', alpha=.5, label=f'1/N = {1/N:.3f}')
ax2.axvline(1.0, color='gray', ls=':', alpha=.5)
ax2.set(xlabel='Fitness r', ylabel='\u03c1 (log scale)', title='Log scale -- deleterious mutants visible')
ax2.set_yscale('log')
ax2.legend()

plt.tight_layout(); plt.show()

In [ ]:
# Isothermal theorem: regular graphs have the same fixation probability as well-mixed
pd.DataFrame([{
    'r': r,
    'well_mixed': f'{exact.well_mixed(N, r):.10f}',
    'isothermal_regular': f'{exact.isothermal_regular(N, r):.10f}',
    'match': exact.well_mixed(N, r) == exact.isothermal_regular(N, r),
} for r in [0.5, 1.0, 1.5, 2.0, 5.0]])

In [ ]:
# try_exact() returns FixationResult for regular graphs, None otherwise
print(f'Cycle (regular):   {exact.try_exact(cycle_graph(20), 1.5)}')
print(f'Star  (irregular): {exact.try_exact(star_graph(20), 1.5)}')

## 3. Population size effect

As $N \to \infty$, the well-mixed fixation probability converges to $1 - 1/r$ for $r > 1$.

In [ ]:
r = 1.5
N_range = np.arange(2, 101)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(N_range, [exact.well_mixed(int(n), r) for n in N_range], 'b-', lw=2)
ax.plot(N_range, [1/n for n in N_range], 'r--', alpha=.5, label='1/N')
ax.axhline(1 - 1/r, color='g', ls=':', alpha=.5, label=f'limit 1-1/r = {1-1/r:.3f}')
ax.set(xlabel='N', ylabel='Fixation probability',
       title=f'Exact well-mixed: convergence to 1-1/r (r={r})')
ax.legend(); plt.tight_layout(); plt.show()

## 4. Naive MC vs exact

Complete and cycle are regular (isothermal) so naive MC should match the exact formula.
Star and double star are irregular -- no exact formula, MC is the only option.

In [ ]:
N = 15
r_grid = np.linspace(0.5, 3.5, 12)
exact_wm = [exact.well_mixed(N, float(r)) for r in r_grid]

graphs = {
    'complete': complete_graph(N),
    'cycle':    cycle_graph(N),
    'star':     star_graph(N),
    'dbl_star': double_star_graph(N // 2, N - N // 2),
}

mc = {}
for name, g in graphs.items():
    mc[name] = [naive.estimate(g, float(r), epsilon=0.1, seed=42, num_threads=1).estimate
                for r in r_grid]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(r_grid, exact_wm, 'k-', lw=2.5, label='Exact well-mixed', zorder=5)
for name, est in mc.items():
    ax.plot(r_grid, est, 'o-', ms=5, color=COLORS[name], label=f'Naive MC: {name}')
ax.axhline(1/N, color='gray', ls='--', alpha=.4, label='1/N')
ax.set(xlabel='r', ylabel='Fixation probability',
       title=f'Naive MC vs exact (N={N}, eps=0.1) -- BD update rule')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

**Why is relative error high at $r = 0.5$?** The Diaz FPRAS guarantees *multiplicative* error
$|\hat\rho - \rho| \le \varepsilon \cdot \rho$, which is only meaningful when $\rho$ is not
exponentially small. For $r < 1$ (deleterious mutants), $\rho \approx r^N \to 0$
exponentially in $N$, so even a tiny additive error produces a huge relative error.

In [ ]:
# Numerical comparison table
N = 15
comp_rows = []
for r in [0.5, 1.0, 1.5, 2.0, 3.0]:
    ex = exact.well_mixed(N, r)
    mc_K = naive.estimate(complete_graph(N), r, epsilon=0.1, seed=42, num_threads=1)
    mc_S = naive.estimate(star_graph(N), r, epsilon=0.1, seed=42, num_threads=1)
    mc_D = naive.estimate(double_star_graph(N // 2, N - N // 2), r, epsilon=0.1, seed=42, num_threads=1)
    comp_rows.append({
        'r': r, 'exact_wm': f'{ex:.6f}',
        'MC_complete': f'{mc_K.estimate:.6f}',
        'MC_star': f'{mc_S.estimate:.6f}',
        'MC_dbl_star': f'{mc_D.estimate:.6f}',
    })
pd.DataFrame(comp_rows)

## 5. Epsilon sweep -- cost vs accuracy

In [ ]:
g = complete_graph(20)
r = 1.5
exact_val = exact.well_mixed(20, r)

eps_grid = [0.30, 0.20, 0.15, 0.10, 0.07]
eps_rows = []
for eps in eps_grid:
    res = naive.estimate(g, r, epsilon=eps, seed=42, num_threads=1)
    eps_rows.append({'epsilon': eps, 'estimate': res.estimate,
                     'ci_lower': res.ci_lower, 'ci_upper': res.ci_upper,
                     'samples': res.samples, 'steps_total': res.steps_total,
                     'time_s': f'{res.elapsed_seconds:.3f}',
                     'rel_error': f'{abs(res.estimate - exact_val) / exact_val:.2%}'})

df_eps = pd.DataFrame(eps_rows)
df_eps

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(df_eps['epsilon'], df_eps['estimate'], 'bo-', label='estimate', zorder=5)
ax1.fill_between(df_eps['epsilon'], df_eps['ci_lower'], df_eps['ci_upper'],
                 alpha=.15, color='b')
ax1.axhline(exact_val, color='r', ls='--', lw=1.5, label=f'exact = {exact_val:.4f}')
ax1.set(xlabel='epsilon', ylabel='fixation probability')
ax1.invert_xaxis()

ax2 = ax1.twinx()
ax2.plot(df_eps['epsilon'], [float(t) for t in df_eps['time_s']], 'rs--', alpha=.7, label='time (s)')
ax2.set_ylabel('time (s)', color='r')

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, fontsize=8)
ax1.set_title('Tighter eps -> more samples -> longer runtime')
plt.tight_layout(); plt.show()

## 6. Ineffective steps -- the Achilles' heel of naive MC

If reproducer and replaced vertex have the same type, the population state is unchanged --
the step is *ineffective* (wasted). Naive MC simulates all of them.
The double star is particularly bad: high-degree hubs create many same-type interactions.

In [ ]:
sizes = [8, 12, 16, 20, 25]
graph_types = ['complete', 'cycle', 'star', 'dbl_star']

def make_graph(name, n):
    if name == 'complete':  return complete_graph(n)
    if name == 'cycle':     return cycle_graph(n)
    if name == 'star':      return star_graph(n)
    if name == 'dbl_star':  return double_star_graph(n // 2, n - n // 2)

eff_rows = []
for n in sizes:
    for name in graph_types:
        g = make_graph(name, n)
        res = naive.estimate(g, 1.5, epsilon=0.15, seed=42, num_threads=1)
        wasted = 1 - res.steps_effective / max(res.steps_total, 1)
        eff_rows.append({'graph': name, 'N': n,
                         'steps_total': res.steps_total,
                         'steps_effective': res.steps_effective,
                         'wasted_frac': wasted, 'time_s': res.elapsed_seconds})

df_eff = pd.DataFrame(eff_rows)
df_eff.pivot_table(index='N', columns='graph', values='wasted_frac').round(3)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: total vs effective steps for double star
sub = df_eff[df_eff['graph'] == 'dbl_star']
x = np.arange(len(sub))
ax1.bar(x - 0.15, sub['steps_total'], 0.3, color='C0', label='total steps')
ax1.bar(x + 0.15, sub['steps_effective'], 0.3, color='C2', label='effective steps')
ax1.set_xticks(x, labels=sub['N'].values)
ax1.set(xlabel='N', ylabel='Steps', title='Double star: total vs effective')
ax1.legend()

# Right: wasted fraction across graph types
for name in graph_types:
    sub = df_eff[df_eff['graph'] == name]
    ax2.plot(sub['N'], sub['wasted_frac'], 'o-', color=COLORS[name],
             lw=2, ms=6, label=name)
ax2.set(xlabel='N', ylabel='Fraction of wasted steps',
        title='Naive MC: wasted steps grow with N', ylim=(0, 1))
ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 7. Runtime scaling

In [ ]:
# Time vs N for different graph structures
fig, ax = plt.subplots(figsize=(7, 4.5))
for name in graph_types:
    sub = df_eff[df_eff['graph'] == name]
    ax.plot(sub['N'], sub['time_s'], 'o-', color=COLORS[name],
            lw=2, ms=6, label=name)
ax.set(xlabel='N', ylabel='Time (s)',
       title='Naive MC runtime vs population size (eps=0.15, r=1.5)')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Time vs r -- structure matters
r_grid_time = [0.5, 1.0, 1.5, 2.0, 3.0]
N = 15

time_rows = []
for r in r_grid_time:
    for name in graph_types:
        g = make_graph(name, N)
        res = naive.estimate(g, r, epsilon=0.15, seed=42, num_threads=1)
        time_rows.append({'r': r, 'graph': name, 'time_s': res.elapsed_seconds,
                          'steps_total': res.steps_total})

df_time_r = pd.DataFrame(time_rows)

fig, ax = plt.subplots(figsize=(7, 4))
for name in graph_types:
    sub = df_time_r[df_time_r['graph'] == name]
    ax.plot(sub['r'], sub['time_s'], 'o-', color=COLORS[name], lw=2, ms=6, label=name)
ax.set(xlabel='r', ylabel='Time (s)',
       title=f'Naive MC runtime vs fitness r (N={N}, eps=0.15)')
ax.legend(); plt.tight_layout(); plt.show()

## 8. Thread scaling

In [ ]:
max_threads = min(multiprocessing.cpu_count(), 16)
thread_counts = sorted(set([1, 2, 4, max_threads]))

g = complete_graph(25)

thread_rows = []
for t in thread_counts:
    res = naive.estimate(g, 1.5, epsilon=0.1, seed=42, num_threads=t)
    thread_rows.append({'threads': t, 'time_s': res.elapsed_seconds,
                        'samples': res.samples, 'estimate': f'{res.estimate:.6f}'})

df_threads = pd.DataFrame(thread_rows)
t1 = df_threads.iloc[0]['time_s']
df_threads['speedup'] = t1 / df_threads['time_s']
df_threads['efficiency'] = df_threads['speedup'] / df_threads['threads']
df_threads

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(df_threads['threads'], df_threads['speedup'], 'bo-', lw=2, ms=8, label='actual')
ax1.plot(df_threads['threads'], df_threads['threads'], 'k--', alpha=.4, label='ideal linear')
ax1.set(xlabel='Threads', ylabel='Speedup', title='Speedup vs threads')
ax1.legend()

ax2.bar(range(len(thread_counts)), df_threads['efficiency'], color='#2ca02c')
ax2.set_xticks(range(len(thread_counts)), labels=thread_counts)
ax2.axhline(1.0, color='k', ls='--', alpha=.3)
ax2.set(xlabel='Threads', ylabel='Efficiency (speedup / threads)',
        title='Parallel efficiency', ylim=(0, 1.2))
plt.suptitle(f'Thread scaling -- K_25, r=1.5, eps=0.1', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

## 9. Amplifiers of selection (BD update rule)

Under Birth-Death dynamics, the **star** and **double star** are **amplifiers** of selection
(Lieberman, Hauert & Nowak, *Nature* 2005): they increase fixation probability
of advantageous mutants ($r > 1$) compared to well-mixed populations.
The double star is one of the strongest known amplifiers (Goldberg 2020).

> Under Death-Birth (DB) dynamics, these graphs are instead *suppressors*.
> This library implements BD.

In [ ]:
N = 20
r_grid = np.linspace(0.5, 4.0, 12)
exact_wm = [exact.well_mixed(N, float(r)) for r in r_grid]

star_est = [naive.estimate(star_graph(N), float(r), epsilon=0.15,
                           seed=42, num_threads=1).estimate for r in r_grid]
dbl_est  = [naive.estimate(double_star_graph(N // 2, N - N // 2), float(r), epsilon=0.15,
                           seed=42, num_threads=1).estimate for r in r_grid]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(r_grid, exact_wm, 'k-', lw=2.5, label='Exact well-mixed', zorder=5)
ax.plot(r_grid, star_est, 'o-', ms=5, color=COLORS['star'], label='Star (amplifier)')
ax.plot(r_grid, dbl_est, 's-', ms=5, color=COLORS['dbl_star'], label='Double star (amplifier)')
ax.axhline(1/N, color='gray', ls='--', alpha=.4, label='1/N')
ax.set(xlabel='r', ylabel='Fixation probability',
       title=f'Amplifiers of selection under BD (N={N})')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## 10. NetworkX / SciPy interop

CSRGraph converts to and from NetworkX graphs and SciPy sparse matrices.
This enables using any NetworkX generator with libmoran.

In [ ]:
# Petersen graph -- classic 3-regular graph
G_pet = nx.petersen_graph()
g_petersen = CSRGraph.from_networkx(G_pet)
print(f'Petersen: {g_petersen}, {g_petersen.degree_stats()}')
print(f'rho(r=1.5) = {fixation_probability(g_petersen, 1.5).estimate:.10f}  (exact, isothermal)')
print()

# Watts-Strogatz small-world graph
G_ws = nx.watts_strogatz_graph(20, 4, 0.3, seed=42)
g_ws = CSRGraph.from_networkx(G_ws)
res_ws = naive.estimate(g_ws, 1.5, epsilon=0.15, seed=42, num_threads=1)
print(f'Watts-Strogatz(20,4,0.3): {g_ws}')
print(f'rho(r=1.5) = {res_ws.estimate:.6f}  [{res_ws.method}]')
print()

# Barabasi-Albert preferential attachment
G_ba = nx.barabasi_albert_graph(20, 2, seed=42)
g_ba = CSRGraph.from_networkx(G_ba)
res_ba = naive.estimate(g_ba, 1.5, epsilon=0.15, seed=42, num_threads=1)
print(f'Barabasi-Albert(20,2): {g_ba}')
print(f'rho(r=1.5) = {res_ba.estimate:.6f}  [{res_ba.method}]')

In [ ]:
# SciPy sparse roundtrip
import scipy.sparse as sp

A = g_petersen.to_scipy_sparse()
print(f'SciPy shape: {A.shape}, nnz: {A.nnz}, symmetric: {(A != A.T).nnz == 0}')

g_rt = CSRGraph.from_scipy_sparse(A)
print(f'Roundtrip:   {g_rt}, edges match: {g_rt.num_edges() == g_petersen.num_edges()}')

## 11. Auto-selection API teaser

`fixation_probability()` picks the best algorithm automatically:
$r=1$ uses exact $1/N$, regular graphs use the isothermal formula,
everything else gets the Goldberg FPRAS (improved over Chatterjee).

In [ ]:
cases = [
    ('K_20, r=1.0',     complete_graph(20), 1.0),
    ('cycle_20, r=1.5', cycle_graph(20),    1.5),
    ('star_15, r=2.0',  star_graph(15),     2.0),
]

auto_rows = []
for label, g, r in cases:
    res = fixation_probability(g, r, epsilon=0.1, seed=42)
    auto_rows.append({'case': label, 'estimate': f'{res.estimate:.6f}',
                      'method': str(res.method).split('.')[-1],
                      'samples': res.samples,
                      'time_s': f'{res.elapsed_seconds:.4f}'})
pd.DataFrame(auto_rows)

---

## Appendix A: FixationResult API reference

In [ ]:
res = naive.estimate(complete_graph(20), 2.0, epsilon=0.1, seed=42, num_threads=1)
print(res)
print()
for attr in ['estimate', 'ci_lower', 'ci_upper', 'method', 'error_model',
             'epsilon', 'delta', 'samples', 'steps_total', 'steps_effective',
             'runs_aborted', 'elapsed_seconds', 'seed_used']:
    print(f'  {attr:20s} = {getattr(res, attr)}')
print()
pd.DataFrame([res.to_dict()])

## Appendix B: Error handling

In [ ]:
from moran import InvalidInputError

cases = []

try: naive.estimate(complete_graph(10), 0.0)
except InvalidInputError as e: cases.append(('r = 0', str(e)))

try: naive.estimate(complete_graph(10), float('nan'))
except InvalidInputError as e: cases.append(('r = NaN', str(e)))

try:
    src = np.array([0, 2], dtype=np.uint32)
    dst = np.array([1, 3], dtype=np.uint32)
    naive.estimate(CSRGraph(4, src, dst), 1.5)
except InvalidInputError as e: cases.append(('disconnected', str(e)))

try: naive.estimate(complete_graph(10), 1.5, epsilon=0.0)
except ValueError as e: cases.append(('eps = 0', str(e)))

try: naive.estimate(complete_graph(10), 1.5, delta=0.0)
except ValueError as e: cases.append(('delta = 0', str(e)))

pd.DataFrame(cases, columns=['trigger', 'message'])